# Sparse Autoencoders: Learning Interpretable Features

This notebook introduces **sparse autoencoders**, an extension of regular autoencoders that learns **sparse**, **interpretable** representations by encouraging only a small fraction of neurons to be active at any time. We'll understand why sparsity matters, implement multiple sparsity constraints, and explore what sparse models learn.

## Learning Objectives

By the end of this notebook, you will:
- Understand what sparsity means and why it's beneficial for representation learning
- Implement sparse autoencoders with KL divergence and L1 regularization
- Compare sparse vs dense learned features visually
- Build overcomplete sparse autoencoders (latent dim > input dim)
- Analyze activation distributions and sparsity levels
- Understand connections to compressed sensing and feature learning

## 1. Introduction: What is Sparsity?

Imagine describing an image using only 5 out of 100 possible features. This is **sparse representation**.

**Dense representation:** All neurons contribute
```
[0.23, 0.19, 0.45, 0.31, 0.28, 0.42, ...]  ← All values non-zero
```

**Sparse representation:** Only a few neurons active
```
[0.00, 0.87, 0.00, 0.00, 0.00, 0.93, ...]  ← Mostly zeros!
```

### Why Sparse Representations?

**Biological motivation:**
- Visual cortex neurons are **sparse** - only ~1% active at once
- Efficient coding theory: sparse codes minimize energy while preserving information

**Machine learning benefits:**
1. **Interpretability**: Each feature has clear meaning (edge detector, circle detector, etc.)
2. **Better generalization**: Forces the model to learn fundamental patterns
3. **Disentangled features**: Features are more independent
4. **Efficient computation**: Most activations are zero
5. **Overcomplete representations**: Can have MORE features than input dimensions

### Sparse vs Regular Autoencoders

**Regular autoencoder:**
- Uses bottleneck (latent_dim < input_dim) to force compression
- All latent neurons typically active
- Features may be entangled

**Sparse autoencoder:**
- Can use bottleneck OR overcomplete (latent_dim > input_dim)
- Adds sparsity constraint: only k% of neurons active
- Forces each neuron to specialize
- Learns more interpretable features

**Key insight:** Sparsity is an alternative (or complement) to the bottleneck for preventing trivial identity mappings!

## 2. Setup and Configuration

### Configuration

Let's define all hyperparameters in one place for easy experimentation.

In [ ]:
CONFIG = {
    # Reproducibility
    'seed': 42,  # Random seed for reproducibility
    
    # Data
    'batch_size': 128,  # Number of samples per training batch
    'num_workers': 0,  # Number of worker processes for data loading
    
    # Model Architecture
    'input_dim': 784,  # MNIST: 28x28 = 784
    'hidden_dim': 512,  # Hidden layer size
    'latent_dim': 256,  # Latent/code layer size (can be > input_dim for overcomplete)
    
    # Training
    'learning_rate': 1e-3,  # Optimizer learning rate
    'num_epochs': 50,  # Number of training epochs
    
    # Sparsity Parameters
    'sparsity_target': 0.01, #0.05 # Target average activation (5% sparsity)
    'sparsity_weight': 3.0,  # Weight for KL divergence sparsity penalty
    'l1_weight': 1e-4,  # Weight for L1 regularization
}

print("Configuration loaded:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

### Import Libraries and Setup

We'll import PyTorch and utilities for building and training our sparse autoencoder.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Shared utilities
from aiml_notebooks import get_device, set_seed

%load_ext autoreload
%autoreload 2

print("✓ Libraries imported successfully")

### Set Random Seed and Device

Ensure reproducibility and use GPU if available.

In [ ]:
set_seed(CONFIG['seed'])
device = get_device()

print(f"Random seed: {CONFIG['seed']}")
print(f"Device: {device}")

### Load MNIST Dataset

We'll use MNIST handwritten digits to learn sparse features.

In [ ]:
# Transform: Convert to tensor and normalize to [0, 1]
transform = transforms.Compose([
    transforms.ToTensor(),
])

# Load datasets
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=CONFIG['num_workers'])
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=CONFIG['num_workers'])

print(f"Training samples: {len(train_dataset):,}")
print(f"Test samples: {len(test_dataset):,}")
print(f"Batch size: {CONFIG['batch_size']}")

### Visualize Sample Images

Let's see some examples from the dataset.

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i, ax in enumerate(axes.flat):
    img, label = train_dataset[i]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(f"{label}", fontsize=10)
    ax.axis('off')
plt.suptitle('Sample MNIST Digits', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f"Each image is 28×28 pixels = {CONFIG['input_dim']} dimensions")

## 3. Sparsity Enforcement Methods

There are several ways to encourage sparsity. We'll implement the two most common.

### Method 1: KL Divergence Penalty

**Idea:** Encourage each neuron's average activation to match a target sparsity level.

For each neuron $j$:
- Let $\hat{\rho}_j$ = average activation across all samples
- Let $\rho$ = target sparsity (e.g., 0.05 = 5%)
- Add penalty: $\text{KL}(\rho \| \hat{\rho}_j) = \rho \log\frac{\rho}{\hat{\rho}_j} + (1-\rho)\log\frac{1-\rho}{1-\hat{\rho}_j}$

This penalizes neurons that are too active OR too inactive!

In [ ]:
def kl_divergence_sparsity(rho_hat, rho=0.05, eps=1e-8):
    """
    Compute KL divergence penalty for sparsity.
    
    Args:
        rho_hat: Average activation per neuron (shape: [latent_dim])
        rho: Target sparsity level
        eps: Small constant for numerical stability
    
    Returns:
        KL divergence penalty (scalar)
    """
    rho_hat = torch.clamp(rho_hat, eps, 1 - eps)  # Avoid log(0)
    
    kl = rho * torch.log(rho / rho_hat) + (1 - rho) * torch.log((1 - rho) / (1 - rho_hat))
    return kl.sum()

print("✓ KL divergence sparsity function defined")

### Method 2: L1 Regularization

**Idea:** Penalize the absolute values of activations directly.

$$\mathcal{L}_{L1} = \lambda \sum_j |z_j|$$

This encourages activations to be exactly zero (sparse).

**L1 vs KL divergence:**
- **L1**: Simpler, encourages exact zeros, works per sample
- **KL**: Encourages consistent sparsity across dataset, more principled

In [ ]:
def l1_penalty(activations):
    """
    Compute L1 penalty on activations.
    
    Args:
        activations: Latent activations (shape: [batch_size, latent_dim])
    
    Returns:
        L1 penalty (scalar)
    """
    return torch.abs(activations).sum(dim=1).mean()

print("✓ L1 penalty function defined")

## 4. Building a Sparse Autoencoder

### Architecture Design

Our sparse autoencoder will have:

**Encoder:**
```
Input (784) → Linear(512) → ReLU → Linear(256) → Sigmoid
                                                     ↑
                                                  Sparse!
```

**Decoder:**
```
Latent (256) → Linear(512) → ReLU → Linear(784) → Sigmoid
```

**Key differences from regular autoencoder:**
1. Sigmoid activation on latent layer (keeps values in [0, 1] for KL divergence)
2. Sparsity penalty added to loss
3. Can use overcomplete representation (256 > 784 is possible!)

### Implement Sparse Autoencoder

Let's build the model with support for both sparsity methods.

In [ ]:
class SparseAutoencoder(nn.Module):
    """Sparse autoencoder with KL divergence and/or L1 sparsity."""
    
    def __init__(self, input_dim=784, hidden_dim=512, latent_dim=256):
        super(SparseAutoencoder, self).__init__()
        
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.latent_dim = latent_dim
        
        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim),
            nn.Sigmoid()  # Sigmoid for sparsity (outputs in [0, 1])
        )
        
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
            nn.Sigmoid()  # Output in [0, 1]
        )
    
    def encode(self, x):
        """Encode input to sparse latent representation."""
        x = x.view(x.size(0), -1)  # Flatten
        return self.encoder(x)
    
    def decode(self, z):
        """Decode latent representation to reconstruction."""
        x = self.decoder(z)
        return x.view(x.size(0), 1, 28, 28)  # Reshape to image
    
    def forward(self, x):
        """Full forward pass."""
        z = self.encode(x)
        x_recon = self.decode(z)
        return x_recon, z

print("✓ SparseAutoencoder class defined")

### Create and Inspect the Model

Let's instantiate our sparse autoencoder.

In [ ]:
model = SparseAutoencoder(
    input_dim=CONFIG['input_dim'],
    hidden_dim=CONFIG['hidden_dim'],
    latent_dim=CONFIG['latent_dim']
).to(device)

print(model)
print(f"\nModel parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"\nLatent dimension: {CONFIG['latent_dim']}")
print(f"Input dimension: {CONFIG['input_dim']}")
print(f"Ratio: {CONFIG['latent_dim'] / CONFIG['input_dim']:.2f}× (overcomplete: {CONFIG['latent_dim'] > CONFIG['input_dim']})")

### Test Forward Pass

Verify the model works correctly.

In [ ]:
# Get a batch
sample_batch, _ = next(iter(train_loader))
sample_batch = sample_batch.to(device)

# Forward pass
with torch.no_grad():
    reconstruction, latent = model(sample_batch)

print(f"Input shape:          {sample_batch.shape}")
print(f"Latent shape:         {latent.shape}")
print(f"Reconstruction shape: {reconstruction.shape}")
print(f"\nLatent activation range: [{latent.min().item():.4f}, {latent.max().item():.4f}]")
print(f"Mean latent activation: {latent.mean().item():.4f}")
print(f"\n✓ Forward pass successful!")

## 5. Training with Sparsity Constraints

### Define Loss Function with Sparsity

Our total loss combines three terms:

$$\mathcal{L}_{\text{total}} = \mathcal{L}_{\text{recon}} + \beta \mathcal{L}_{\text{KL}} + \lambda \mathcal{L}_{\text{L1}}$$

Where:
- $\mathcal{L}_{\text{recon}}$ = reconstruction error (BCE)
- $\mathcal{L}_{\text{KL}}$ = KL divergence sparsity penalty
- $\mathcal{L}_{\text{L1}}$ = L1 regularization
- $\beta, \lambda$ = weight hyperparameters

### Implement Training Functions

We'll track reconstruction loss and sparsity separately.

In [ ]:
def train_epoch(model, train_loader, optimizer, device, config):
    """Train sparse autoencoder for one epoch."""
    model.train()
    total_loss = 0
    total_recon_loss = 0
    total_sparsity_loss = 0
    
    # Collect activations for KL divergence (average over epoch)
    all_activations = []
    
    for batch_idx, (data, _) in enumerate(train_loader):
        data = data.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass
        reconstruction, latent = model(data)
        
        # Reconstruction loss (BCE)
        recon_loss = F.binary_cross_entropy(reconstruction, data, reduction='mean')
        
        # L1 sparsity penalty
        l1_loss = l1_penalty(latent)
        
        # Store activations for KL divergence
        all_activations.append(latent.detach())
        
        # Total loss (we'll add KL at the end)
        loss = recon_loss + config['l1_weight'] * l1_loss
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        total_recon_loss += recon_loss.item()
        total_sparsity_loss += l1_loss.item()
    
    # Compute KL divergence over all activations
    all_activations = torch.cat(all_activations, dim=0)
    rho_hat = all_activations.mean(dim=0)  # Average activation per neuron
    kl_loss = kl_divergence_sparsity(rho_hat, rho=config['sparsity_target'])
    
    n_batches = len(train_loader)
    return (
        total_loss / n_batches,
        total_recon_loss / n_batches,
        total_sparsity_loss / n_batches,
        kl_loss.item(),
        rho_hat.cpu().numpy()
    )

def evaluate(model, test_loader, device):
    """Evaluate sparse autoencoder on test set."""
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        for data, _ in test_loader:
            data = data.to(device)
            reconstruction, _ = model(data)
            loss = F.binary_cross_entropy(reconstruction, data, reduction='mean')
            total_loss += loss.item()
    
    return total_loss / len(test_loader)

print("✓ Training functions defined")

### Train the Sparse Autoencoder

Now let's train our model with sparsity constraints!

In [ ]:
# Create optimizer
optimizer = optim.Adam(model.parameters(), lr=CONFIG['learning_rate'])

# Track metrics
train_losses = []
recon_losses = []
sparsity_losses = []
kl_losses = []
test_losses = []
activation_history = []

# Training loop
print(f"Training for {CONFIG['num_epochs']} epochs...\n")
for epoch in tqdm(range(CONFIG['num_epochs']), desc="Training"):
    train_loss, recon_loss, sparsity_loss, kl_loss, rho_hat = train_epoch(
        model, train_loader, optimizer, device, CONFIG
    )
    test_loss = evaluate(model, test_loader, device)
    
    train_losses.append(train_loss)
    recon_losses.append(recon_loss)
    sparsity_losses.append(sparsity_loss)
    kl_losses.append(kl_loss)
    test_losses.append(test_loss)
    activation_history.append(rho_hat)
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:2d}/{CONFIG['num_epochs']} - "
              f"Train: {train_loss:.4f}, Test: {test_loss:.4f}, "
              f"Recon: {recon_loss:.4f}, Sparsity: {sparsity_loss:.4f}, KL: {kl_loss:.4f}")

print("\n✓ Training complete!")

### Visualize Training Progress

Let's see how the different loss components evolved.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Reconstruction loss
axes[0, 0].plot(recon_losses, label='Train Recon', linewidth=2, color='#4ECDC4')
axes[0, 0].plot(test_losses, label='Test Recon', linewidth=2, color='#FF6B6B')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Reconstruction Loss (BCE)')
axes[0, 0].set_title('Reconstruction Loss')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# L1 sparsity loss
axes[0, 1].plot(sparsity_losses, linewidth=2, color='#95E1D3')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('L1 Penalty')
axes[0, 1].set_title('L1 Sparsity Penalty')
axes[0, 1].grid(alpha=0.3)

# KL divergence loss
axes[1, 0].plot(kl_losses, linewidth=2, color='#F38181')
axes[1, 0].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('KL Divergence')
axes[1, 0].set_title('KL Divergence Sparsity Penalty')
axes[1, 0].grid(alpha=0.3)

# Total loss
axes[1, 1].plot(train_losses, label='Total Train', linewidth=2, color='#AA96DA')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Total Loss')
axes[1, 1].set_title('Total Training Loss')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.suptitle('Sparse Autoencoder Training Progress', fontsize=16, y=1.00)
plt.tight_layout()
plt.show()

print(f"Final reconstruction loss: {recon_losses[-1]:.4f}")
print(f"Final L1 penalty: {sparsity_losses[-1]:.6f}")
print(f"Final KL divergence: {kl_losses[-1]:.4f}")

## 6. Analyzing Sparsity

### Visualize Activation Distribution

Let's see if our model learned sparse activations.

In [ ]:
# Get activations on test set
model.eval()
all_latents = []

with torch.no_grad():
    for data, _ in test_loader:
        data = data.to(device)
        _, latent = model(data)
        all_latents.append(latent.cpu().numpy())

all_latents = np.concatenate(all_latents, axis=0)  # Shape: (n_samples, latent_dim)

# Plot histogram of all activations
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# All activations
axes[0].hist(all_latents.flatten(), bins=100, edgecolor='black', alpha=0.7, color='#4ECDC4')
axes[0].axvline(x=CONFIG['sparsity_target'], color='red', linestyle='--', linewidth=2, 
                label=f"Target: {CONFIG['sparsity_target']}")
axes[0].axvline(x=all_latents.mean(), color='orange', linestyle='--', linewidth=2,
                label=f"Actual: {all_latents.mean():.4f}")
axes[0].set_xlabel('Activation Value', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('Distribution of All Activations', fontsize=14)
axes[0].legend()
axes[0].grid(alpha=0.3, axis='y')

# Average activation per neuron
mean_activations = all_latents.mean(axis=0)  # Average across samples
axes[1].hist(mean_activations, bins=50, edgecolor='black', alpha=0.7, color='#95E1D3')
axes[1].axvline(x=CONFIG['sparsity_target'], color='red', linestyle='--', linewidth=2,
                label=f"Target: {CONFIG['sparsity_target']}")
axes[1].set_xlabel('Mean Activation', fontsize=12)
axes[1].set_ylabel('Number of Neurons', fontsize=12)
axes[1].set_title('Average Activation per Neuron', fontsize=14)
axes[1].legend()
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Compute sparsity statistics
threshold = 0.1  # Consider activation < 0.1 as "inactive"
sparsity_per_sample = (all_latents < threshold).mean(axis=1)

print(f"\nSparsity Statistics:")
print(f"  Target sparsity: {CONFIG['sparsity_target']:.4f}")
print(f"  Actual mean activation: {all_latents.mean():.4f}")
print(f"  Fraction of activations < {threshold}: {(all_latents < threshold).mean():.4f}")
print(f"  Average sparsity per sample: {sparsity_per_sample.mean():.4f}")

### Visualize Lifetime Sparsity

**Lifetime sparsity**: How often is each neuron active across all inputs?

Good sparse models should have neurons that are highly selective (only active for specific inputs).

In [ ]:
# Count how many samples activate each neuron
activation_threshold = 0.3  # Consider neuron "active" if > 0.3
active_count = (all_latents > activation_threshold).sum(axis=0)  # Count per neuron
active_fraction = active_count / len(all_latents)

# Sort neurons by activity
sorted_indices = np.argsort(active_fraction)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of lifetime sparsity
axes[0].hist(active_fraction, bins=50, edgecolor='black', alpha=0.7, color='#F38181')
axes[0].axvline(x=active_fraction.mean(), color='blue', linestyle='--', linewidth=2,
                label=f'Mean: {active_fraction.mean():.4f}')
axes[0].set_xlabel('Fraction of Samples Where Neuron is Active', fontsize=12)
axes[0].set_ylabel('Number of Neurons', fontsize=12)
axes[0].set_title('Lifetime Sparsity Distribution', fontsize=14)
axes[0].legend()
axes[0].grid(alpha=0.3, axis='y')

# Activity per neuron (sorted)
axes[1].plot(active_fraction[sorted_indices], linewidth=2, color='#AA96DA')
axes[1].axhline(y=0.05, color='red', linestyle='--', alpha=0.5, label='5% activity')
axes[1].set_xlabel('Neuron Index (sorted by activity)', fontsize=12)
axes[1].set_ylabel('Fraction Active', fontsize=12)
axes[1].set_title('Per-Neuron Activity (Sorted)', fontsize=14)
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nLifetime Sparsity:")
print(f"  Most active neuron: {active_fraction.max():.4f} ({active_count.max()} / {len(all_latents)} samples)")
print(f"  Least active neuron: {active_fraction.min():.4f} ({active_count.min()} / {len(all_latents)} samples)")
print(f"  Mean activity: {active_fraction.mean():.4f}")
print(f"  Neurons active < 5%: {(active_fraction < 0.05).sum()} / {len(active_fraction)}")

## 7. Visualizing Learned Features

### Reconstruction Quality

First, let's see how well the sparse autoencoder reconstructs images.

In [ ]:
def visualize_reconstructions(model, dataset, device, n_samples=10):
    """Display original and reconstructed images."""
    model.eval()
    
    indices = np.random.choice(len(dataset), n_samples, replace=False)
    images = torch.stack([dataset[i][0] for i in indices]).to(device)
    labels = [dataset[i][1] for i in indices]
    
    with torch.no_grad():
        reconstructions, latents = model(images)
    
    # Count active neurons per image
    active_counts = (latents > 0.3).sum(dim=1).cpu().numpy()
    
    fig, axes = plt.subplots(2, n_samples, figsize=(n_samples*1.5, 3.5))
    
    for i in range(n_samples):
        # Original
        axes[0, i].imshow(images[i].cpu().squeeze(), cmap='gray')
        axes[0, i].set_title(f"{labels[i]}", fontsize=10)
        axes[0, i].axis('off')
        if i == 0:
            axes[0, i].set_ylabel('Original', fontsize=12, rotation=0, ha='right', va='center')
        
        # Reconstruction
        axes[1, i].imshow(reconstructions[i].cpu().squeeze(), cmap='gray')
        axes[1, i].set_title(f"{active_counts[i]}/{CONFIG['latent_dim']} active", fontsize=8)
        axes[1, i].axis('off')
        if i == 0:
            axes[1, i].set_ylabel('Reconstructed', fontsize=12, rotation=0, ha='right', va='center')
    
    plt.suptitle('Sparse Autoencoder Reconstructions', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
    
    print(f"Average active neurons: {active_counts.mean():.1f} / {CONFIG['latent_dim']} ({active_counts.mean()/CONFIG['latent_dim']*100:.1f}%)")

visualize_reconstructions(model, test_dataset, device)

### Visualize Decoder Weights (Learned Features)

Each decoder weight represents what pattern that latent neuron encodes. Let's visualize the most and least active features!

In [ ]:
# Get decoder weights (first layer maps from latent to hidden)
# We want the weights that directly generate pixels
decoder_final_weights = model.decoder[-2].weight.data.cpu().numpy()  # Shape: (784, hidden_dim)

# Get the full decoder output for each latent unit
# Create one-hot vectors for each latent dimension
n_features_to_show = 64
feature_indices = np.random.choice(CONFIG['latent_dim'], n_features_to_show, replace=False)

model.eval()
features = []
for idx in feature_indices:
    # Create one-hot latent vector
    z = torch.zeros(1, CONFIG['latent_dim']).to(device)
    z[0, idx] = 1.0  # Activate only this neuron
    
    with torch.no_grad():
        feature = model.decode(z).cpu().squeeze().numpy()
    features.append(feature)

# Plot features in a grid
n_cols = 8
n_rows = n_features_to_show // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols*1.5, n_rows*1.5))
axes = axes.flat

for i, (ax, feature) in enumerate(zip(axes, features)):
    ax.imshow(feature, cmap='gray')
    ax.set_title(f"F{feature_indices[i]}", fontsize=8)
    ax.axis('off')

plt.suptitle('Learned Sparse Features (Random Sample)', fontsize=14, y=0.995)
plt.tight_layout()
plt.show()

print(f"Each image shows what one latent neuron encodes")
print(f"These are the building blocks the model uses to reconstruct digits!")

### Visualize Most and Least Active Features

Let's see what the most frequently used features look like vs the rare ones.

In [ ]:
# Sort neurons by lifetime sparsity (computed earlier)
most_active_indices = np.argsort(active_fraction)[-16:][::-1]  # Top 16
least_active_indices = np.argsort(active_fraction)[:16]  # Bottom 16

def decode_features(model, indices, device):
    """Decode features for given neuron indices."""
    features = []
    for idx in indices:
        z = torch.zeros(1, CONFIG['latent_dim']).to(device)
        z[0, idx] = 1.0
        with torch.no_grad():
            feature = model.decode(z).cpu().squeeze().numpy()
        features.append(feature)
    return features

most_active_features = decode_features(model, most_active_indices, device)
least_active_features = decode_features(model, least_active_indices, device)

# Plot
fig, axes = plt.subplots(2, 16, figsize=(16, 3))

for i in range(16):
    # Most active
    axes[0, i].imshow(most_active_features[i], cmap='gray')
    axes[0, i].set_title(f"{active_fraction[most_active_indices[i]]:.2f}", fontsize=8)
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_ylabel('Most Active', fontsize=10, rotation=0, ha='right', va='center')
    
    # Least active
    axes[1, i].imshow(least_active_features[i], cmap='gray')
    axes[1, i].set_title(f"{active_fraction[least_active_indices[i]]:.2f}", fontsize=8)
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_ylabel('Least Active', fontsize=10, rotation=0, ha='right', va='center')

plt.suptitle('Most vs Least Active Learned Features', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Title shows fraction of samples where neuron is active")
print("Notice: Most active features are often more general/common patterns")
print("Least active features are more specific/rare patterns")

## 8. Understanding What Neurons Learn

### Find Selective Neurons

Let's find neurons that activate strongly for specific digit classes.

In [ ]:
# Get latents with labels
model.eval()
latents_by_class = {i: [] for i in range(10)}

with torch.no_grad():
    for data, labels in test_loader:
        data = data.to(device)
        _, latent = model(data)
        latent = latent.cpu().numpy()
        labels = labels.numpy()
        
        for i in range(len(labels)):
            latents_by_class[labels[i]].append(latent[i])

# Stack by class
latents_by_class = {k: np.array(v) for k, v in latents_by_class.items()}

# Compute mean activation per neuron per class
mean_activation_per_class = np.array([latents_by_class[i].mean(axis=0) for i in range(10)])  # (10, latent_dim)

# Find most selective neurons (highest variance across classes)
selectivity = mean_activation_per_class.std(axis=0)  # Higher = more selective
most_selective = np.argsort(selectivity)[-8:][::-1]

# Visualize
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flat

for i, neuron_idx in enumerate(most_selective):
    ax = axes[i]
    activations = mean_activation_per_class[:, neuron_idx]
    
    bars = ax.bar(range(10), activations, color='skyblue', edgecolor='black')
    # Highlight max
    max_class = np.argmax(activations)
    bars[max_class].set_color('red')
    
    ax.set_xlabel('Digit Class', fontsize=10)
    ax.set_ylabel('Mean Activation', fontsize=10)
    ax.set_title(f'Neuron {neuron_idx} (σ={selectivity[neuron_idx]:.3f})', fontsize=11)
    ax.set_xticks(range(10))
    ax.grid(alpha=0.3, axis='y')

plt.suptitle('Most Selective Neurons: Activation by Digit Class', fontsize=14, y=1.00)
plt.tight_layout()
plt.show()

print("Each plot shows how strongly a neuron activates for each digit (0-9)")
print("Red bar = digit that most activates this neuron")
print("High selectivity = neuron specializes in one or few digits")

### Visualize Maximum Activating Images

For a few neurons, let's see which images activate them the most.

In [ ]:
# Pick 4 interesting neurons (e.g., most selective)
neurons_to_analyze = most_selective[:4]

fig, axes = plt.subplots(len(neurons_to_analyze), 8, figsize=(12, len(neurons_to_analyze)*1.5))

for row, neuron_idx in enumerate(neurons_to_analyze):
    # Find top 8 activating images for this neuron
    neuron_activations = all_latents[:, neuron_idx]  # all_latents from earlier
    top_indices = np.argsort(neuron_activations)[-8:][::-1]
    
    for col, idx in enumerate(top_indices):
        ax = axes[row, col] if len(neurons_to_analyze) > 1 else axes[col]
        
        img, label = test_dataset[idx]
        ax.imshow(img.squeeze(), cmap='gray')
        ax.set_title(f"{label} ({neuron_activations[idx]:.2f})", fontsize=8)
        ax.axis('off')
        
        if col == 0:
            ax.set_ylabel(f'N{neuron_idx}', fontsize=10, rotation=0, ha='right', va='center')

plt.suptitle('Top Activating Images per Neuron', fontsize=14, y=0.995)
plt.tight_layout()
plt.show()

print("Each row shows the 8 images that most activate a specific neuron")
print("Title: (digit label, activation value)")

## 9. Comparing Sparse vs Dense Autoencoders

### Train a Dense Autoencoder for Comparison

Let's train a regular (non-sparse) autoencoder with the same architecture to compare.

In [ ]:
# Create dense autoencoder (no sparsity constraints)
dense_model = SparseAutoencoder(
    input_dim=CONFIG['input_dim'],
    hidden_dim=CONFIG['hidden_dim'],
    latent_dim=CONFIG['latent_dim']
).to(device)

dense_optimizer = optim.Adam(dense_model.parameters(), lr=CONFIG['learning_rate'])

# Train without sparsity penalties
def train_dense_epoch(model, train_loader, optimizer, device):
    model.train()
    total_loss = 0
    for data, _ in train_loader:
        data = data.to(device)
        optimizer.zero_grad()
        reconstruction, _ = model(data)
        loss = F.binary_cross_entropy(reconstruction, data, reduction='mean')
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)

print("Training dense autoencoder (no sparsity)...")
dense_losses = []
for epoch in tqdm(range(CONFIG['num_epochs']), desc="Training dense"):
    loss = train_dense_epoch(dense_model, train_loader, dense_optimizer, device)
    dense_losses.append(loss)

print("✓ Dense autoencoder trained")

### Compare Activation Distributions

Let's see the difference in sparsity between models.

In [ ]:
# Get activations from dense model
dense_model.eval()
dense_latents = []

with torch.no_grad():
    for data, _ in test_loader:
        data = data.to(device)
        _, latent = dense_model(data)
        dense_latents.append(latent.cpu().numpy())

dense_latents = np.concatenate(dense_latents, axis=0)

# Compare distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sparse model
axes[0].hist(all_latents.flatten(), bins=100, alpha=0.7, label='Sparse AE', 
             color='#4ECDC4', edgecolor='black')
axes[0].axvline(x=all_latents.mean(), color='blue', linestyle='--', linewidth=2,
                label=f'Mean: {all_latents.mean():.4f}')
axes[0].set_xlabel('Activation Value', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('Sparse Autoencoder', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3, axis='y')

# Dense model
axes[1].hist(dense_latents.flatten(), bins=100, alpha=0.7, label='Dense AE',
             color='#FF6B6B', edgecolor='black')
axes[1].axvline(x=dense_latents.mean(), color='red', linestyle='--', linewidth=2,
                label=f'Mean: {dense_latents.mean():.4f}')
axes[1].set_xlabel('Activation Value', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_title('Dense Autoencoder (No Sparsity)', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3, axis='y')

plt.suptitle('Activation Distribution: Sparse vs Dense', fontsize=16, y=1.00)
plt.tight_layout()
plt.show()

# Statistics
sparse_near_zero = (all_latents < 0.1).mean()
dense_near_zero = (dense_latents < 0.1).mean()

print(f"\nSparsity Comparison:")
print(f"  Sparse AE - Mean activation: {all_latents.mean():.4f}, Fraction < 0.1: {sparse_near_zero:.4f}")
print(f"  Dense AE  - Mean activation: {dense_latents.mean():.4f}, Fraction < 0.1: {dense_near_zero:.4f}")
print(f"\n  → Sparse model has {sparse_near_zero/dense_near_zero:.2f}× more near-zero activations!")

### Compare Learned Features

Visualize features from both models side by side.

In [ ]:
# Select same random features from both models
n_compare = 16
compare_indices = np.random.choice(CONFIG['latent_dim'], n_compare, replace=False)

sparse_features = decode_features(model, compare_indices, device)
dense_features = decode_features(dense_model, compare_indices, device)

# Plot side by side
fig, axes = plt.subplots(2, n_compare, figsize=(n_compare*1.2, 3))

for i in range(n_compare):
    # Sparse
    axes[0, i].imshow(sparse_features[i], cmap='gray')
    axes[0, i].axis('off')
    if i == 0:
        axes[0, i].set_ylabel('Sparse', fontsize=12, rotation=0, ha='right', va='center')
    
    # Dense
    axes[1, i].imshow(dense_features[i], cmap='gray')
    axes[1, i].axis('off')
    if i == 0:
        axes[1, i].set_ylabel('Dense', fontsize=12, rotation=0, ha='right', va='center')

plt.suptitle('Learned Features: Sparse vs Dense Autoencoders', fontsize=14, y=0.98)
plt.tight_layout()
plt.show()

print("Sparse features often appear more localized and interpretable")
print("Dense features may be more distributed/entangled")

## 10. Key Takeaways

### Core Concepts

1. **Sparse representations** use only a small fraction of neurons at a time, mimicking biological neural networks

2. **Sparsity enforcement** can be achieved through:
   - **KL divergence penalty**: Encourages consistent average activation per neuron
   - **L1 regularization**: Penalizes absolute activation values

3. **Benefits of sparsity**:
   - More interpretable features (each neuron specializes)
   - Better generalization (forces learning of fundamental patterns)
   - Enables overcomplete representations (more features than input dims)
   - More disentangled features

4. **Overcomplete autoencoders** (latent_dim > input_dim) only work with sparsity constraints - otherwise they learn identity mapping

5. **Lifetime sparsity** measures how selective each neuron is across all inputs

6. **Sparse features** are often more localized and interpretable than dense features

7. **Selectivity**: Sparse autoencoders naturally learn neurons that respond strongly to specific patterns or classes

### Connections to Other Concepts

**Sparse autoencoders relate to:**

- **Compressed sensing**: Sparse representations enable signal reconstruction from fewer measurements
- **Dictionary learning**: Learning overcomplete bases for representing signals
- **Neuroscience**: Efficient coding theory and sparse coding in visual cortex
- **Regularization**: L1 penalty is also used in Lasso regression for feature selection
- **Disentangled representations**: Sparsity encourages independence of features
- **Interpretability**: Sparse models are easier to understand and analyze
- **VAEs with sparse priors**: Can use sparse distributions (e.g., Laplace) as priors

### Experiments to Try

**1. Vary sparsity target:**
- Try `sparsity_target` = 0.01 (very sparse) vs 0.2 (less sparse)
- How does it affect reconstruction quality and feature interpretability?

**2. Overcomplete representations:**
- Set `latent_dim` = 1024 (much larger than 784 input)
- Does the model learn more fine-grained features?

**3. Compare L1 vs KL:**
- Train one model with only L1, another with only KL
- Which produces better features?

**4. Convolutional sparse autoencoder:**
- Replace linear layers with Conv2d
- Do you learn edge detectors and Gabor-like filters?

**5. Alternative activation functions:**
- Try ReLU instead of Sigmoid on latent layer
- ReLU naturally produces exact zeros (perfect sparsity)

**6. Transfer to classification:**
- Freeze the sparse encoder and add a classifier on top
- Do sparse features transfer better than dense?

## Summary

Congratulations! You've mastered sparse autoencoders:

✅ Understood what sparsity means and why it's beneficial  
✅ Implemented sparsity constraints using KL divergence and L1 regularization  
✅ Built and trained overcomplete sparse autoencoders  
✅ Analyzed activation distributions and lifetime sparsity  
✅ Visualized learned features and found selective neurons  
✅ Compared sparse vs dense representations empirically  

Sparse autoencoders are a powerful tool for learning interpretable, disentangled representations. They bridge classical ideas from neuroscience and compressed sensing with modern deep learning, and continue to be relevant in research on interpretable AI and feature learning!

**Next steps:** Explore Variational Autoencoders (VAEs) for probabilistic sparse representations, or apply sparse autoencoders to other domains like audio or text.